# BeforeTheNotice feasibility check
Run preparation and matching first. Record source coverage, audit decisions and the go/no-go threshold in docs/data_decisions.md.

In [ ]:
from pathlib import Path
import pandas as pd

root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
processed = root / 'data' / 'processed'
reviews = pd.read_csv(processed / 'reviews.csv', parse_dates=['review_date'])
warn = pd.read_csv(processed / 'warn.csv', parse_dates=['notice_date'])
display(reviews[['review_date']].agg(['min', 'max']))
display(warn[['notice_date']].agg(['min', 'max']))
counts = reviews.groupby('company_key').size().rename('reviews')
print('Review employers:', counts.size)
display(counts.describe(), counts.sort_values(ascending=False).head(20))

In [ ]:
candidates = pd.read_csv(processed / 'match_candidates.csv')
display(candidates.method.value_counts())
audit = pd.read_csv(processed / 'match_audit_sample.csv')
display(audit.head())
print('Review the audit CSV and resolve employer links before measuring eligible overlap.')

In [ ]:
# Use reviewed links and a reproducible six-month window before a candidate forecast date.
def eligible_matched(cutoff, minimum=5):
    cutoff = pd.Timestamp(cutoff)
    window = reviews.review_date.between(cutoff - pd.DateOffset(months=6), cutoff, inclusive='left')
    counts = reviews[window].groupby('company_key').size()
    links = pd.read_csv(processed / 'reviewed_links.csv')
    companies = pd.read_csv(processed / 'companies.csv')
    eligible = companies[companies.company_key.isin(counts[counts.ge(minimum)].index)]
    return eligible[eligible.company_key.isin(links.review_key)].workforce_group.value_counts()